In [1]:
import random
import pandas as pd
import json

In [2]:
# Vocabulary
shapes = ['circle', 'square', 'triangle', 'rectangle', 'oval', 
          'pentagon', 'hexagon', 'diamond']

colours = ['red', 'blue', 'green', 'yellow', 'orange', 
           'purple', 'pink', 'brown', 'grey']

sizes = ['small', 'medium', 'large']

relations = ['above', 'below', 'to the left of', 'to the right of', 'next to', 'inside']

In [3]:
# Sentence templates
templates = [
    "a {size1} {colour1} {shape1} is {relation} a {size2} {colour2} {shape2}",
    "there is a {size1} {colour1} {shape1} {relation} a {size2} {colour2} {shape2}",
    "one {size1} {colour1} {shape1} is {relation} one {size2} {colour2} {shape2}",
    "{article1} {colour1} {shape1} is {relation} {article2} {colour2} {shape2}",
    "a {size1} {shape1} is {relation} a {size2} {shape2}",
    "a {size1} {colour1} {shape1} is positioned {relation} a {size2} {colour2} {shape2}"
]

In [6]:
def get_article(word):
    if word[0].lower() in ['a', 'e', 'i', 'o', 'u']:
        return 'an'
    return 'a'

def generate_sentence(template, shape1, colour1, size1, relation, shape2, colour2, size2):
    article1 = get_article(colour1)
    article2 = get_article(colour2)
    return template.format(
        article1=article1, article2=article2,
        shape1=shape1, colour1=colour1, size1=size1,
        relation=relation,
        shape2=shape2, colour2=colour2, size2=size2
    )

# Generating sentences
sentences = []
sentence_id = 1

for i in range(4000):
    template = random.choice(templates)
    shape1 = random.choice(shapes)
    shape2 = random.choice(shapes)
    colour1 = random.choice(colours)
    colour2 = random.choice(colours)
    relation = random.choice(relations)

    if relation == 'inside':
        size1 = random.choice(['small', 'medium'])
        if size1 == 'small':
            size2 = random.choice(['medium', 'large'])
        else:  # size1 is medium
            size2 = 'large'
    else:
        size1 = random.choice(sizes)
        size2 = random.choice(sizes)
    
    sentence = generate_sentence(template, shape1, colour1, size1, 
                                 relation, shape2, colour2, size2)
    
    sentences.append({
        'sentence_id': sentence_id,
        'sentence': sentence,
        'obj1_shape': shape1,
        'obj1_colour': colour1,
        'obj1_size': size1,
        'relation': relation,
        'obj2_shape': shape2,
        'obj2_colour': colour2,
        'obj2_size': size2,
        'template_id': templates.index(template) + 1
    })
    
    sentence_id += 1

print(f"Generated {len(sentences)} sentences")
print("\nExample sentences:")
for s in random.sample(sentences, 20):
    print(s['sentence'])

Generated 4000 sentences

Example sentences:
there is a small pink rectangle below a small brown oval
a small hexagon is next to a large square
there is a small blue triangle to the right of a small yellow circle
a small diamond is below a medium circle
there is a large blue circle to the left of a small grey triangle
one large pink circle is next to one large blue diamond
a red square is to the right of a purple oval
there is a large blue circle to the left of a medium blue pentagon
a small brown diamond is to the right of a large yellow pentagon
one large orange circle is below one small pink hexagon
one medium purple triangle is to the left of one medium brown circle
a small orange hexagon is positioned next to a large purple diamond
a large pentagon is below a large diamond
there is a small purple hexagon above a medium brown diamond
a grey pentagon is to the right of a grey circle
one medium purple circle is to the right of one medium blue triangle
a medium pink triangle is positi

In [10]:
# Checking duplicates
df = pd.DataFrame(sentences)
duplicates = df[df.duplicated(subset=['sentence'])]
print(f"Duplicate sentences: {len(duplicates)}")

Duplicate sentences: 105


In [11]:
# Removing duplicates
df = df.drop_duplicates(subset=['sentence']).reset_index(drop=True)

# Updating sentence IDs
df['sentence_id'] = range(1, len(df) + 1)

In [12]:
print("\nFirst 5 rows:")
print(df.head())
print("\nDataset columns:", list(df.columns))


First 5 rows:
   sentence_id                                           sentence obj1_shape  \
0            1           a large square is above a small triangle     square   
1            2  there is a small pink hexagon inside a large g...    hexagon   
2            3  a medium red rectangle is below a medium brown...  rectangle   
3            4      an orange triangle is above an orange diamond   triangle   
4            5  a small orange hexagon is to the right of a sm...    hexagon   

  obj1_colour obj1_size         relation obj2_shape obj2_colour obj2_size  \
0         red     large            above   triangle         red     small   
1        pink     small           inside  rectangle        grey     large   
2         red    medium            below     square       brown    medium   
3      orange     small            above    diamond      orange     large   
4      orange     small  to the right of   pentagon       brown     small   

   template_id  
0            5  
1      

In [13]:
print("Total sentences:", len(df))
print("\nTemplate distribution:")
print(df['template_id'].value_counts().sort_index())
print("\nShape distribution:")
print(df['obj1_shape'].value_counts())

Total sentences: 3895

Template distribution:
template_id
1    679
2    649
3    672
4    657
5    621
6    617
Name: count, dtype: int64

Shape distribution:
obj1_shape
circle       522
hexagon      505
triangle     501
pentagon     499
square       486
rectangle    484
oval         452
diamond      446
Name: count, dtype: int64


In [14]:
# Saving cleaned dataset
df.to_csv('sentence_dataset_shapes.csv', index=False)

print(f"Cleaned dataset: {len(df)} sentences")

Cleaned dataset: 3895 sentences
